In [ ]:
# -*- coding: utf-8 -*-

import os
import glob
import warnings
from os.path import join as pjoin

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'
from tqdm.auto import trange
import xarray as xr
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

from utils import plt_color_dir


def sig_stars(pval):
    """Convert p-value to significance stars"""
    if pval < 0.001:
        return '***'
    elif pval < 0.01:
        return '**'
    elif pval < 0.05:
        return '*'
    else:
        return ''


def analyze_tau_values(save_dir, areas_all, n_back):
    """Analyze tau values and preserve mouse ID information"""
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    all_data = []
    
    for area_idx in trange(len(areas_all), desc='Processing areas'):
        area_name = area_names[area_idx]
        sessions_all = areas_all[area_idx]
        
        for ss in range(len(sessions_all)):
            session_name = sessions_all[ss]
            mouse_id = session_name[:5]
            
            try:
                output_xarray = xr.open_dataset(pjoin(save_dir, 'cellfits_halves_compare_mdl', 
                                                     'sse_{}_{}hist_updated.nc'.format(session_name, n_back)))
                
                crit_RCp1 = output_xarray.sel(mdl_type='Rp1', half=0).p_beta_RC < 0.05
                crit_sig_half1 = output_xarray.sel(mdl_type='exp_r', half=1).p_beta_RC < 0.05
                crit_sig_half2 = output_xarray.sel(mdl_type='exp_r', half=2).p_beta_RC < 0.05
                
                out_pd = output_xarray.sel(half=0, mdl_type='exp_r').to_dataframe()[['beta_RC', 'tau_RC']]
                out_pd['crit_RCp1'] = crit_RCp1
                out_pd['crit_sig_half1'] = crit_sig_half1
                out_pd['crit_sig_half2'] = crit_sig_half2
                
                filtered_taus = out_pd.loc[(out_pd['crit_sig_half1'] == True) & 
                                (out_pd['crit_sig_half2'] == True) & 
                                (out_pd['tau_RC'] < 99.9) & 
                                (out_pd['tau_RC'] >= 0.11), 'tau_RC']
                
                for tau in filtered_taus:
                    all_data.append({
                        'mouse_id': mouse_id,
                        'area': area_name,
                        'session': session_name,
                        'tau': tau
                    })
            except:
                pass
    
    return pd.DataFrame(all_data)


def analyze_beta_values(save_dir, areas_all, n_back):
    """Analyze beta values and preserve mouse ID information"""
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    all_data = []
    
    for area_idx in trange(len(areas_all), desc='Processing areas'):
        area_name = area_names[area_idx]
        sessions_all = areas_all[area_idx]
        
        for ss in range(len(sessions_all)):
            session_name = sessions_all[ss]
            mouse_id = session_name[:5]
            
            try:
                output_xarray = xr.open_dataset(pjoin(save_dir, 'cellfits_halves_compare_mdl', 
                                                     'sse_{}_{}hist_updated.nc'.format(session_name, n_back)))
                
                crit_RCp1 = output_xarray.sel(mdl_type='Rp1', half=0).p_beta_RC < 0.05
                crit_sig_half1 = output_xarray.sel(mdl_type='exp_r', half=1).p_beta_RC < 0.05
                crit_sig_half2 = output_xarray.sel(mdl_type='exp_r', half=2).p_beta_RC < 0.05
                
                out_pd = output_xarray.sel(half=0, mdl_type='exp_r').to_dataframe()[['beta_RC', 'tau_RC']]
                out_pd['crit_RCp1'] = crit_RCp1
                out_pd['crit_sig_half1'] = crit_sig_half1
                out_pd['crit_sig_half2'] = crit_sig_half2
                
                filtered_betas = out_pd.loc[(out_pd['crit_sig_half1'] == True) & 
                                          (out_pd['crit_sig_half2'] == True) & 
                                          (out_pd['tau_RC'] < 99.9) & 
                                          (out_pd['tau_RC'] >= 0.11), 'beta_RC']
                
                for beta in filtered_betas:
                    all_data.append({
                        'mouse_id': mouse_id,
                        'area': area_name,
                        'session': session_name,
                        'beta': beta
                    })
            except:
                pass
    
    return pd.DataFrame(all_data)


def create_session_medians(df_short, df_long, value_col):
    """Calculate median values for each session"""
    session_medians = []
    
    for condition, df in [('short', df_short), ('long', df_long)]:
        for area in ['RSC', 'PPC', 'M2', 'S1']:
            area_data = df[df['area'] == area]
            for session in area_data['session'].unique():
                session_data = area_data[area_data['session'] == session]
                mouse_id = session_data['mouse_id'].iloc[0]
                
                if value_col == 'beta':
                    median_val = np.abs(session_data[value_col]).median()
                else:
                    median_val = session_data[value_col].median()
                
                session_medians.append({
                    'mouse_id': mouse_id,
                    'area': area,
                    'condition': condition,
                    'session': session,
                    f'median_{value_col}': median_val
                })
    
    median_df = pd.DataFrame(session_medians)
    median_df['group'] = median_df['area'] + '_' + median_df['condition']
    return median_df


def run_mixed_effects_analysis(median_df, value_col):
    """Run mixed effects models for each brain region"""
    mixed_effect_results = {}
    
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        area_data = median_df[median_df['area'] == area]
        
        try:
            model = smf.mixedlm(
                f"median_{value_col} ~ condition", 
                area_data, 
                groups=area_data["mouse_id"]
            )
            model_fit = model.fit(reml=True)
            
            param_names = list(model_fit.pvalues.index)
            condition_params = [p for p in param_names if 'condition' in p.lower()]
            
            if condition_params:
                p_val = model_fit.pvalues[condition_params[0]]
                effect_size = model_fit.params[condition_params[0]]
                mixed_effect_results[area] = {
                    'p_value': p_val,
                    'effect_size': effect_size,
                    'converged': True
                }
        except:
            mixed_effect_results[area] = {'converged': False}
    
    # Apply FDR correction
    pvals = []
    areas = []
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        if area in mixed_effect_results and 'p_value' in mixed_effect_results[area]:
            pvals.append(mixed_effect_results[area]['p_value'])
            areas.append(area)
    
    if pvals:
        _, pvals_corrected, _, _ = multipletests(pvals, method='fdr_bh')
        for i, area in enumerate(areas):
            mixed_effect_results[area]['p_value_raw'] = mixed_effect_results[area]['p_value']
            mixed_effect_results[area]['p_value'] = pvals_corrected[i]
    
    return mixed_effect_results


def plot_barplot_with_mouse_colors(median_df, save_dir, value_col, fig_number, mixed_effects_results):
    """Create barplot with dots colored by mouse ID"""
    plt_colors = plt_color_dir()
    
    # Create color map for mice
    all_mice = median_df['mouse_id'].unique()
    n_mice = len(all_mice)
    mouse_colors = plt.cm.tab20(np.linspace(0, 1, n_mice))
    mouse_color_map = {mouse: mouse_colors[i] for i, mouse in enumerate(all_mice)}
    
    # Calculate group statistics
    group_stats = {}
    for area in ['RSC', 'PPC', 'M2', 'S1']:
        for condition in ['short', 'long']:
            group = f"{area}_{condition}"
            group_data = median_df[median_df['group'] == group]
            
            if len(group_data) > 0:
                group_stats[group] = {
                    'mean': group_data[f'median_{value_col}'].mean(),
                    'sem': stats.sem(group_data[f'median_{value_col}'].values),
                    'data': group_data[f'median_{value_col}'].values,
                    'mouse_ids': group_data['mouse_id'].values
                }
            else:
                group_stats[group] = {
                    'mean': np.nan,
                    'sem': np.nan,
                    'data': np.array([]),
                    'mouse_ids': np.array([])
                }
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Setup bar positions
    bar_width = 0.35
    index = np.arange(4)
    area_names = ['RSC', 'PPC', 'M2', 'S1']
    
    # Plot bars
    short_means = [group_stats[f'{area}_short']['mean'] for area in area_names]
    short_sems = [group_stats[f'{area}_short']['sem'] for area in area_names]
    long_means = [group_stats[f'{area}_long']['mean'] for area in area_names]
    long_sems = [group_stats[f'{area}_long']['sem'] for area in area_names]
    
    ax.bar(index - bar_width/2, short_means, bar_width, yerr=short_sems,
           color=[plt_colors[area] for area in area_names],
           alpha=0.7, edgecolor='black', linewidth=1,
           error_kw={'capsize': 3, 'elinewidth': 1, 'capthick': 1},
           label='Short Task')
    
    ax.bar(index + bar_width/2, long_means, bar_width, yerr=long_sems,
           color=[plt_colors[area] for area in area_names],
           alpha=0.3, edgecolor='black', linewidth=1,
           error_kw={'capsize': 3, 'elinewidth': 1, 'capthick': 1},
           label='Long Task')
    
    # Add scatter points with mouse-colored fill and edges
    for i, area in enumerate(area_names):
        # Short condition
        short_data = group_stats[f'{area}_short']['data']
        short_mice = group_stats[f'{area}_short']['mouse_ids']
        if len(short_data) > 0:
            short_jitter = np.random.normal(0, 0.03, size=len(short_data))
            for j, (val, mouse) in enumerate(zip(short_data, short_mice)):
                ax.scatter(i - bar_width/2 + short_jitter[j], val, 
                          color=mouse_color_map[mouse], s=50, alpha=0.8, 
                          edgecolors=mouse_color_map[mouse], linewidths=1.5, zorder=3)
        
        # Long condition
        long_data = group_stats[f'{area}_long']['data']
        long_mice = group_stats[f'{area}_long']['mouse_ids']
        if len(long_data) > 0:
            long_jitter = np.random.normal(0, 0.03, size=len(long_data))
            for j, (val, mouse) in enumerate(zip(long_data, long_mice)):
                ax.scatter(i + bar_width/2 + long_jitter[j], val,
                          color=mouse_color_map[mouse], s=50, alpha=0.8,
                          edgecolors=mouse_color_map[mouse], linewidths=1.5, zorder=3)
    
    # Add significance markers
    if value_col == 'tau':
        y_bar = 4.5
        y_text = 4.7
        ax.set_ylim([0, 5])
        ylabel = 'Median τ value'
    else:
        y_bar = 0.18
        y_text = 0.19
        ax.set_ylim([0, 0.2])
        ylabel = 'Median |β| value'
    
    if mixed_effects_results:
        for i, area in enumerate(area_names):
            if area in mixed_effects_results and 'p_value' in mixed_effects_results[area]:
                p_val = mixed_effects_results[area]['p_value']
                stars = sig_stars(p_val)
                
                if stars:
                    ax.plot([i - bar_width/2, i + bar_width/2], [y_bar, y_bar], 'k-', linewidth=1)
                    ax.text(i, y_text, stars, ha='center', fontsize=12)
    
    # Customize plot
    ax.set_xlabel('Brain Region', fontsize=14)
    ax.set_ylabel(ylabel, fontsize=14)
    title = f'Figure {fig_number}: {"Tau" if value_col == "tau" else "Beta"} Adaptation by Region (Short vs Long Task)'
    ax.set_title(title, fontsize=16)
    ax.set_xticks(index)
    ax.set_xticklabels(area_names, fontsize=12)
    ax.legend(fontsize=12)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.figtext(0.5, 0.01, "* p<0.05, ** p<0.01, *** p<0.001\n(FDR-corrected)", 
               ha='center', fontsize=10, bbox=dict(facecolor='lightyellow', alpha=0.5))
    
    # Save figure
    plt.tight_layout()
    filename = f'Figure{fig_number}_{value_col}_adaptation_barplot_mouse_colors'
    fig.savefig(os.path.join(save_dir, f'{filename}.png'), dpi=300)
    fig.savefig(os.path.join(save_dir, f'{filename}.svg'))
    plt.close()


def load_session_data(params, task_type='long'):
    """Load session data and organize by brain region"""
    mice_dataset = params['long_mice'] if task_type == 'long' else params['short_mice']
    data_dir = params['main_dir'] + mice_dataset
    
    os.chdir(data_dir)
    list_of_files = glob.glob('*.mat')
    
    sessions_RSC = [s[0:12] for s in list_of_files if 'RSC' in s]
    sessions_PPC = [s[0:12] for s in list_of_files if 'PPC' in s]
    sessions_M2 = [s[0:12] for s in list_of_files if 'M2' in s]
    sessions_S1 = [s[0:12] for s in list_of_files if 'S1' in s]
    
    return [sessions_RSC, sessions_PPC, sessions_M2, sessions_S1]


# Main execution
if __name__ == "__main__":
    params = {
        'main_dir': 'C:/Users/Eva/Desktop/behavior250629/',
        'save_dir': 'C:/Users/Eva/Desktop/Manuscript2025/Figures/3',
        'imaging_dir': 'C:/Users/Eva/Desktop/imaging250629',
        'long_mice': 'long_all_imaging',
        'short_mice': 'short_all_imaging',
        'n_back': 15
    }
    
    os.makedirs(params['save_dir'], exist_ok=True)
    
    # Load session data
    areas_all_long = load_session_data(params, 'long')
    areas_all_short = load_session_data(params, 'short')
    
    # Process tau values
    df_tau_long = analyze_tau_values(params['imaging_dir'], areas_all_long, params['n_back'])
    df_tau_short = analyze_tau_values(params['imaging_dir'], areas_all_short, params['n_back'])
    
    # Process beta values
    df_beta_long = analyze_beta_values(params['imaging_dir'], areas_all_long, params['n_back'])
    df_beta_short = analyze_beta_values(params['imaging_dir'], areas_all_short, params['n_back'])
    
    # Create session-level medians for tau
    median_df_tau = create_session_medians(df_tau_short, df_tau_long, 'tau')
    mixed_results_tau = run_mixed_effects_analysis(median_df_tau, 'tau')
    plot_barplot_with_mouse_colors(median_df_tau, params['save_dir'], 'tau', 5, mixed_results_tau)
    
    # Create session-level medians for beta
    median_df_beta = create_session_medians(df_beta_short, df_beta_long, 'beta')
    mixed_results_beta = run_mixed_effects_analysis(median_df_beta, 'beta')
    plot_barplot_with_mouse_colors(median_df_beta, params['save_dir'], 'beta', 6, mixed_results_beta)